# Gradient Descent & Optimization

---

## 1. The Core Idea

You have a **loss function** $\mathcal{L}(\theta)$ and you want to find the parameters $\theta$ that minimize it.

Gradient descent answers: *in which direction should I nudge $\theta$ to make $\mathcal{L}$ smaller?*

The answer: **the opposite direction of the gradient**.

$$\theta_{t+1} = \theta_t - \eta \cdot \nabla_\theta \mathcal{L}(\theta_t)$$

| Symbol | Meaning |
|--------|---------|
| $\theta_t$ | Parameters at step $t$ |
| $\eta$ | Learning rate (step size) |
| $\nabla_\theta \mathcal{L}$ | Gradient — vector of partial derivatives |

## 2. What Is a Gradient?

For $\mathcal{L}: \mathbb{R}^n \to \mathbb{R}$, the gradient is the vector of all partial derivatives:

$$\nabla_\theta \mathcal{L} = \begin{pmatrix} \frac{\partial \mathcal{L}}{\partial \theta_1} \\ \frac{\partial \mathcal{L}}{\partial \theta_2} \\ \vdots \\ \frac{\partial \mathcal{L}}{\partial \theta_n} \end{pmatrix}$$

Each component $\frac{\partial \mathcal{L}}{\partial \theta_i}$ answers: *if I increase $\theta_i$ by a tiny $\varepsilon$, how much does the loss change?*

The gradient points in the direction of **steepest ascent**. We go the opposite direction → steepest descent.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.patches import FancyArrowPatch

plt.rcParams.update({
    'figure.facecolor': '#0f0f0f',
    'axes.facecolor': '#1a1a1a',
    'axes.edgecolor': '#444',
    'axes.labelcolor': '#ccc',
    'xtick.color': '#888',
    'ytick.color': '#888',
    'text.color': '#eee',
    'grid.color': '#2a2a2a',
    'grid.linewidth': 0.8,
    'font.family': 'monospace',
    'axes.titlesize': 13,
    'axes.labelsize': 11,
})
ACCENT = '#00e5ff'
ORANGE = '#ff6d00'
GREEN  = '#69ff47'
RED    = '#ff4d6d'

In [ ]:
# --- Gradient descent on a simple 1D convex loss ---
def loss(x): return (x - 2)**2 + 1
def grad(x): return 2 * (x - 2)

def run_gd(start, lr, steps=15):
    path = [start]
    x = start
    for _ in range(steps):
        x = x - lr * grad(x)
        path.append(x)
    return path

x_range = np.linspace(-3, 7, 400)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle('Effect of Learning Rate $\\eta$ on Convergence', fontsize=14, color='#eee', y=1.02)

configs = [
    (0.05, '$\\eta = 0.05$ — too slow',    ACCENT),
    (0.9,  '$\\eta = 0.9$  — good',         GREEN),
    (1.2,  '$\\eta = 1.2$  — diverges',     RED),
]

for ax, (lr, title, color) in zip(axes, configs):
    path = run_gd(start=-2, lr=lr, steps=20)
    ax.plot(x_range, loss(x_range), color='#555', lw=2, label='$\\mathcal{L}(\\theta)$')
    xs = np.array(path)
    ys = loss(xs)
    ax.scatter(xs, ys, color=color, s=40, zorder=5)
    ax.plot(xs, ys, color=color, lw=1, alpha=0.6)
    # Annotate start and last point
    ax.annotate('start', (xs[0], ys[0]), textcoords='offset points', xytext=(8, 8),
                color='#aaa', fontsize=9)
    ax.axvline(x=2, color='#555', linestyle='--', lw=1, label='minimum')
    ax.set_title(title, color=color, fontsize=11)
    ax.set_xlabel('$\\theta$')
    ax.set_ylabel('$\\mathcal{L}(\\theta)$')
    ax.grid(True)
    ax.set_ylim(-0.5, 30)
    ax.set_xlim(-3, 7)

plt.tight_layout()
plt.savefig('img/01_learning_rate.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. Convexity

A function is **convex** if the line segment between any two points on its graph lies *above* the graph:

$$\mathcal{L}(\lambda a + (1-\lambda) b) \leq \lambda \mathcal{L}(a) + (1-\lambda)\mathcal{L}(b), \quad \lambda \in [0,1]$$

**Why it matters**: a convex loss has a **unique global minimum**. Gradient descent is guaranteed to find it.

The second-order condition: $\mathcal{L}$ is convex iff its **Hessian** $H = \nabla^2 \mathcal{L}$ is **positive semi-definite** — all eigenvalues $\geq 0$:
$$v^\top H v \geq 0 \quad \forall v \in \mathbb{R}^n$$

| Model | Loss | Convex? |
|-------|------|---------|
| Linear Regression | MSE | ✅ |
| Logistic Regression | Log-loss | ✅ |
| Neural Networks | Cross-entropy / MSE | ❌ (non-convex) |

In [ ]:
# --- Convex vs non-convex loss surface ---
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('Loss Surface: Convex vs Non-Convex', fontsize=14, color='#eee')

x = np.linspace(-4, 4, 500)

# Convex
ax = axes[0]
y_convex = x**2 + 1
ax.plot(x, y_convex, color=GREEN, lw=2.5)
# Show convexity: chord between two points
a, b = -2.5, 2.5
ax.plot([a, b], [a**2+1, b**2+1], color=ORANGE, lw=2, linestyle='--', label='chord (always above)')
ax.scatter([2], [5], color=ACCENT, s=100, zorder=5, label='global minimum')
ax.axvline(0, color='#555', lw=1, linestyle=':')
ax.set_title('Convex — one global minimum', color=GREEN)
ax.set_xlabel('$\\theta$'); ax.set_ylabel('$\\mathcal{L}(\\theta)$')
ax.legend(fontsize=9); ax.grid(True)

# Non-convex
ax = axes[1]
y_nonconvex = np.sin(2*x) * np.exp(-0.1*x**2) * 3 + 0.3*x**2
ax.plot(x, y_nonconvex, color=ORANGE, lw=2.5)
# Local minima
local_mins_x = [-2.0, 0.5, 2.8]
for lm in local_mins_x:
    idx = np.argmin(np.abs(x - lm))
    ax.scatter(x[idx], y_nonconvex[idx], color=RED, s=80, zorder=5)
# Global min
gmin_idx = np.argmin(y_nonconvex)
ax.scatter(x[gmin_idx], y_nonconvex[gmin_idx], color=ACCENT, s=120, zorder=6,
           label='global min', marker='*')
ax.scatter([], [], color=RED, s=80, label='local minima / saddle pts')
ax.set_title('Non-Convex — local minima & saddle points', color=ORANGE)
ax.set_xlabel('$\\theta$'); ax.set_ylabel('$\\mathcal{L}(\\theta)$')
ax.legend(fontsize=9); ax.grid(True)

plt.tight_layout()
plt.savefig('img/02_convexity.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Three Flavors of Gradient Descent

The difference is *how many samples* you use to estimate the gradient at each step.

### Batch GD (BGD)
Use the **entire dataset** of $n$ samples:
$$\nabla_\theta \mathcal{L} = \frac{1}{n} \sum_{i=1}^{n} \nabla_\theta \ell(\theta; x_i, y_i)$$
✅ Stable, exact gradient — ❌ one update = full pass over data

### Stochastic GD (SGD)
Use a **single random sample**:
$$\theta_{t+1} = \theta_t - \eta \cdot \nabla_\theta \ell(\theta_t; x_i, y_i)$$
✅ Very fast updates, noisy enough to escape local minima — ❌ high variance, unstable convergence

### Mini-Batch GD (the standard)
Use a **batch of $B$ samples** ($B \in \{32, 64, 128, 256\}$):
$$\theta_{t+1} = \theta_t - \frac{\eta}{B} \sum_{i \in \mathcal{B}} \nabla_\theta \ell(\theta_t; x_i, y_i)$$
✅ Vectorized efficiency + controlled noise — the default in deep learning

In [ ]:
# --- BGD vs SGD vs Mini-batch: loss curves ---
np.random.seed(42)
steps = 100
t = np.arange(steps)

# Simulate convergence curves
bgd   = 10 * np.exp(-0.06 * t) + 0.5
sgd   = 10 * np.exp(-0.05 * t) + 0.5 + np.random.normal(0, 0.8, steps) * np.exp(-0.02*t)
mini  = 10 * np.exp(-0.055 * t) + 0.5 + np.random.normal(0, 0.25, steps) * np.exp(-0.02*t)

fig, ax = plt.subplots(figsize=(11, 5))
ax.plot(t, bgd,  color=GREEN,  lw=2.5, label='Batch GD — smooth, slow per epoch')
ax.plot(t, sgd,  color=RED,    lw=1.5, alpha=0.85, label='SGD — fast but noisy')
ax.plot(t, mini, color=ACCENT, lw=2,   alpha=0.9,  label='Mini-Batch GD — best trade-off')
ax.set_xlabel('Iteration'); ax.set_ylabel('$\\mathcal{L}$')
ax.set_title('Loss Curve: BGD vs SGD vs Mini-Batch', fontsize=13)
ax.legend(fontsize=10); ax.grid(True)
plt.tight_layout()
plt.savefig('img/03_gd_variants.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Why the Learning Rate $\eta$ Matters

**Taylor expansion** — why gradient descent works locally:

$$\mathcal{L}(\theta - \eta g) \approx \mathcal{L}(\theta) - \eta \|g\|^2 + \mathcal{O}(\eta^2)$$

where $g = \nabla_\theta \mathcal{L}(\theta)$. As long as $\eta$ is small enough, the first-order term dominates: the loss **strictly decreases**.

When $\eta$ is too large, the $\mathcal{O}(\eta^2)$ term (which involves curvature) takes over and the update overshoots.

## 6. Advanced Optimizers

Plain gradient descent is rarely used in practice. The key variants:

### Momentum
Accumulates a velocity vector $v$ to dampen oscillations and accelerate in consistent directions:

$$v_{t+1} = \beta v_t + (1-\beta)\, \nabla_\theta \mathcal{L}(\theta_t)$$
$$\theta_{t+1} = \theta_t - \eta \cdot v_{t+1}$$

Typical $\beta = 0.9$. Intuition: a ball rolling down a hill builds speed in consistent directions and resists noise.

### RMSProp
Adapts the learning rate **per parameter** using a running average of squared gradients:

$$s_{t+1} = \beta s_t + (1-\beta)\, g_t^2$$
$$\theta_{t+1} = \theta_t - \frac{\eta}{\sqrt{s_{t+1}} + \varepsilon}\, g_t$$

Parameters with large gradients get a smaller effective $\eta$ — useful in landscapes with varying curvature.

### Adam — the default in deep learning

Combines Momentum (1st moment) + RMSProp (2nd moment):

$$m_{t+1} = \beta_1 m_t + (1-\beta_1)\, g_t \qquad \text{(mean)}$$
$$v_{t+1} = \beta_2 v_t + (1-\beta_2)\, g_t^2 \qquad \text{(variance)}$$

**Bias correction** — crucial in early steps when $m, v \approx 0$:

$$\hat{m} = \frac{m_{t+1}}{1 - \beta_1^{t+1}}, \qquad \hat{v} = \frac{v_{t+1}}{1 - \beta_2^{t+1}}$$

**Update rule**:

$$\boxed{\theta_{t+1} = \theta_t - \frac{\eta}{\sqrt{\hat{v}} + \varepsilon}\, \hat{m}}$$

Defaults: $\beta_1 = 0.9$, $\beta_2 = 0.999$, $\varepsilon = 10^{-8}$

In [ ]:
# --- Optimizer trajectories on a 2D loss surface (saddle + valley) ---
def rosenbrock(x, y, a=1, b=8):
    """Modified Rosenbrock-style valley loss."""
    return (a - x)**2 + b*(y - x**2)**2

def grad_rosenbrock(x, y, a=1, b=8):
    dx = -2*(a - x) - 4*b*x*(y - x**2)
    dy = 2*b*(y - x**2)
    return np.array([dx, dy])

def optimize(grad_fn, start, lr, steps, method='sgd', beta1=0.9, beta2=0.999, eps=1e-8):
    path = [np.array(start, dtype=float)]
    theta = np.array(start, dtype=float)
    m = np.zeros(2); v = np.zeros(2); vel = np.zeros(2)
    for t in range(1, steps+1):
        g = grad_fn(*theta)
        if method == 'sgd':
            theta = theta - lr * g
        elif method == 'momentum':
            vel = beta1 * vel + (1 - beta1) * g
            theta = theta - lr * vel
        elif method == 'adam':
            m = beta1*m + (1-beta1)*g
            v = beta2*v + (1-beta2)*g**2
            mh = m / (1 - beta1**t)
            vh = v / (1 - beta2**t)
            theta = theta - lr * mh / (np.sqrt(vh) + eps)
        path.append(theta.copy())
    return np.array(path)

start = [-1.5, 2.5]
steps = 300

path_sgd  = optimize(grad_rosenbrock, start, lr=0.005, steps=steps, method='sgd')
path_mom  = optimize(grad_rosenbrock, start, lr=0.005, steps=steps, method='momentum')
path_adam = optimize(grad_rosenbrock, start, lr=0.05,  steps=steps, method='adam')

# Loss surface
xi = np.linspace(-2, 2, 300)
yi = np.linspace(-1, 4, 300)
X, Y = np.meshgrid(xi, yi)
Z = rosenbrock(X, Y)

fig, ax = plt.subplots(figsize=(10, 7))
cp = ax.contourf(X, Y, np.log1p(Z), levels=40, cmap='inferno', alpha=0.85)
ax.contour(X, Y, np.log1p(Z), levels=20, colors='white', linewidths=0.3, alpha=0.3)
plt.colorbar(cp, ax=ax, label='log(1 + Loss)')

for path, color, label in [
    (path_sgd,  RED,    'SGD'),
    (path_mom,  ORANGE, 'Momentum'),
    (path_adam, ACCENT, 'Adam'),
]:
    ax.plot(path[:, 0], path[:, 1], color=color, lw=1.8, label=label, alpha=0.9)
    ax.scatter(*path[-1], color=color, s=80, zorder=5)

ax.scatter(1, 1, color='white', s=150, zorder=6, marker='*', label='global min (1,1)')
ax.scatter(*start, color='#aaa', s=80, zorder=6, marker='x', label='start')
ax.set_title('Optimizer Trajectories on a Valley-Shaped Loss Surface', fontsize=13)
ax.set_xlabel('$\\theta_1$'); ax.set_ylabel('$\\theta_2$')
ax.legend(fontsize=10)
plt.tight_layout()
plt.savefig('img/04_optimizer_trajectories.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Loss Surface Pathologies (Non-Convex Case)

In neural networks the loss landscape is complex. Three key obstacles:

### Saddle Points
$\nabla \mathcal{L} = 0$ but the Hessian has **mixed-sign eigenvalues** — the loss decreases in some directions, increases in others. In high dimensions, saddle points are far more common than local minima. SGD noise helps escape them.

### Vanishing Gradients
In deep networks, gradients are chained via the chain rule through $L$ layers. If each layer multiplies by a factor $< 1$ (e.g. sigmoid saturation), the gradient vanishes exponentially:

$$\frac{\partial \mathcal{L}}{\partial \theta_1} = \frac{\partial \mathcal{L}}{\partial z_L} \cdot \prod_{k=1}^{L} \frac{\partial z_k}{\partial z_{k-1}} \xrightarrow{L \to \infty} 0$$

Mitigations: **residual connections** (ResNet), **batch normalization**, **ReLU activations**, **careful initialization** (He, Xavier).

### Exploding Gradients
The same product can blow up if each factor $> 1$. Common in RNNs. Mitigated by **gradient clipping**: $g \leftarrow g \cdot \frac{c}{\max(c, \|g\|)}$.

In [ ]:
# --- Saddle point visualization ---
xi = np.linspace(-2, 2, 200)
yi = np.linspace(-2, 2, 200)
X, Y = np.meshgrid(xi, yi)
Z_saddle = X**2 - Y**2  # classic saddle

fig = plt.figure(figsize=(12, 5))

# 3D surface
ax1 = fig.add_subplot(121, projection='3d')
ax1.set_facecolor('#1a1a1a')
ax1.plot_surface(X, Y, Z_saddle, cmap='coolwarm', alpha=0.85, linewidth=0)
ax1.scatter([0], [0], [0], color='white', s=80, zorder=5)
ax1.set_title('Saddle Point: $\\mathcal{L} = \\theta_1^2 - \\theta_2^2$', color='#eee')
ax1.set_xlabel('$\\theta_1$', color='#aaa')
ax1.set_ylabel('$\\theta_2$', color='#aaa')
ax1.set_zlabel('$\\mathcal{L}$', color='#aaa')
ax1.tick_params(colors='#666')

# 2D contour with gradient arrows
ax2 = fig.add_subplot(122)
cp = ax2.contourf(X, Y, Z_saddle, levels=30, cmap='coolwarm', alpha=0.85)
plt.colorbar(cp, ax=ax2)
# Gradient arrows
skip = 20
Gx = 2 * X[::skip, ::skip]
Gy = -2 * Y[::skip, ::skip]
ax2.quiver(X[::skip, ::skip], Y[::skip, ::skip], -Gx, -Gy,
           color='white', alpha=0.5, scale=40, width=0.003)
ax2.scatter([0], [0], color='white', s=120, zorder=5, marker='*')
ax2.set_title('Gradient field — arrows show $-\\nabla\\mathcal{L}$', color='#eee')
ax2.set_xlabel('$\\theta_1$'); ax2.set_ylabel('$\\theta_2$')
ax2.grid(True)

plt.tight_layout()
plt.savefig('img/05_saddle_point.png', dpi=150, bbox_inches='tight')
plt.show()

## 8. Connections to Other Concepts

| Concept | Link to Gradient Descent |
|---------|-------------------------|
| **Backpropagation** | Algorithm to *compute* $\nabla_\theta \mathcal{L}$ efficiently via the chain rule |
| **L2 Regularization** | Adds $\lambda \|\theta\|^2$ to $\mathcal{L}$ → gradient gains a weight decay term $+2\lambda\theta$ |
| **Learning Rate Schedules** | $\eta$ decreases over training (cosine annealing, step decay, warm-up) |
| **Batch Normalization** | Smooths the loss surface → allows larger $\eta$, faster convergence |
| **Second-Order Methods** | Newton's method uses the Hessian $H^{-1}$ instead of just $\nabla\mathcal{L}$ — exact but $\mathcal{O}(n^3)$ cost |

## Key Takeaways

1. Gradient descent is a **first-order method** — it uses only the gradient, not curvature. This is why the learning rate matters so much.
2. Convexity guarantees a global minimum. Neural networks are non-convex — we rely on empirical results and good initializations.
3. **Adam is the default** in deep learning; SGD with momentum can sometimes generalize better in vision tasks.
4. The gradient lives in **parameter space** — its dimension equals the number of model parameters, not the input size.
5. In high dimensions, **saddle points** are the real enemy, not local minima.

---
*Next: [`02_linear_regression.ipynb`](./02_linear_regression.ipynb)*